In [55]:
# Imports
import json
import concurrent.futures
import re
from textwrap import dedent
from statistics import mean
from dotenv import load_dotenv
from anthropic import Anthropic

In [67]:
# Client Initialization and helper functions

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

from anthropic.types import Message

def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message
    }
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools=None, tool_choice=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    if tool_choice:
        params["tool_choice"] = tool_choice

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join(
        [block.text for block in message.content if block.type == "text"]
    )


def run_batch(invocations=[]):
    batch_ouputs = []

    for invocation in invocations:
        name = invocation["name"]
        args = json.loads(invocation["arguments"])
        tool_ouptut = run_tool(name, args)
        batch_ouptuts.append({
            "tool_name": name,
            "output": tool_output
        })
    return batch_ouputs
    

def run_tool(tool_name, tool_input):
    if tool_name == "get_current_datetime":
        return get_current_datetime(**tool_input)
    elif tool_name == "add_duration_to_datetime":
        return add_duration_to_datetime(**tool_input)
    elif tool_name == "set_reminder":
        return set_reminder(**tool_input)
    elif tool_name == "batch_tool":
        return run_batch(**tool_input)
        

def run_tools(message):
    tool_requests = [
        block for block in message.content if block.type == "tool_use"
    ]

    tool_result_blocks = []

    for tool_request in tool_requests:
        try:
            tool_output = run_tool(tool_request.name, tool_request.input)
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": json.dumps(tool_output),
                "is_error": False
            }
        except Exception as e:
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": f"Error: {e}",
                "is_error": True
            }
        tool_result_blocks.append(tool_result_block)
    return tool_result_blocks


def run_conversation(messages):
    while True:
        response = chat(messages, tools=[
            get_current_datetime_schema,
            add_duration_to_datetime_schema,
            set_reminder_schema,
            batch_tool_schema
        ])

        add_assistant_message(messages, response)
        print(text_from_message(response))

        if response.stop_reason != "tool_use":
            break

        tool_results = run_tools(response)
        add_user_message(messages, tool_results)
    return messages
    

In [64]:
# Tools and Schemas

from datetime import datetime, timedelta

def add_duration_to_datetime(
    datetime_str, duration=0, unit="days", input_format="%Y-%m-%d"
):
    date = datetime.strptime(datetime_str, input_format)

    if unit == "seconds":
        new_date = date + timedelta(seconds=duration)
    elif unit == "minutes":
        new_date = date + timedelta(minutes=duration)
    elif unit == "hours":
        new_date = date + timedelta(hours=duration)
    elif unit == "days":
        new_date = date + timedelta(days=duration)
    elif unit == "weeks":
        new_date = date + timedelta(weeks=duration)
    elif unit == "months":
        month = date.month + duration
        month = month % 12
        if month == 0:
            month = 12
            year = -1
            day = min(
                date.day,
                [
                    31,
                    29 if year % 4 == 0 and (year % 100 != 0 or year % 400 == 0) else 28,
                    31,
                    30,
                    31,
                    30,
                    31,
                    31,
                    30,
                    31,
                    30,
                    31,
                ][month-1]
            )
            new_date = date.replace(year=year, month=month, day=day)
    elif unit == "years":
        new_date = date.replace(year=date.year + duration)
    else:
        raise ValueError(f"Unsupported time unit: {unit}")

    return new_date.strftime("%A, %B %d, %Y %I:%M:%S %p")

def set_reminder(content, timestamp):
    print(
        f"----\n setting the following reminder for {timestamp}:\n{content}\n------"
    )
    

add_duration_to_datetime_schema = {
  "name": "add_duration_to_datetime",
  "description": "Adds a specified duration to a datetime string and returns the resulting date formatted as a readable string.",
  "input_schema": {
    "type": "object",
    "properties": {
      "datetime_str": {
        "type": "string",
        "description": "The input datetime string to add duration to"
      },
      "duration": {
        "type": "integer",
        "description": "The amount of time to add (can be negative)",
        "default": 0
      },
      "unit": {
        "type": "string",
        "enum": ["seconds", "minutes", "hours", "days", "weeks", "months", "years"],
        "description": "The unit of time for the duration",
        "default": "days"
      },
      "input_format": {
        "type": "string",
        "description": "The strptime format of the input datetime string",
        "default": "%Y-%m-%d"
      }
    },
    "required": ["datetime_str"]
  }
}

set_reminder_schema = {
  "name": "set_reminder",
  "description": "Sets a reminder with the specified content for a given timestamp.",
  "input_schema": {
    "type": "object",
    "properties": {
      "content": {
        "type": "string",
        "description": "The reminder message or content"
      },
      "timestamp": {
        "type": "string",
        "description": "When the reminder should trigger"
      }
    },
    "required": ["content", "timestamp"]
  }
}

batch_tool_schema = {
  "name": "batch_tool",
  "description": "Execute multiple datetime and reminder operations in a single call.",
  "input_schema": {
    "type": "object",
    "properties": {
      "invocations": {
        "type": "array",
        "description": "List of operations to execute in sequence",
        "items": {
          "type": "object",
          "properties": {
            "name": {
              "type": "string",
              "description": "The tool to execute"
            },
            "arguments": {
              "type": "string",
              "description": "The arguments to the tool, encoded as a JSON string",
            }
          },
          "required": ["name", "arguments"]
        }
      }
    },
    "required": ["invocations"]
  }
}

article_summary_schema = {
  "name": "article_summary",
  "description": "Summarizes an article with its title, author, and key insights.",
  "input_schema": {
    "type": "object",
    "properties": {
      "title": {
        "type": "string",
        "description": "The title of the article"
      },
      "author": {
        "type": "string",
        "description": "The author of the article"
      },
      "key_insights": {
        "type": "array",
        "items": {
          "type": "string"
        },
        "description": "List of key insights from the article"
      }
    },
    "required": ["title", "author", "key_insights"]
  }
}

In [58]:
from datetime import datetime
from anthropic.types import ToolParam

def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format cannot be empty")
    return datetime.now().strftime(date_format)

get_current_datetime_schema = ToolParam({
  "name": "get_current_datetime",
  "description": "Get the current date and time formatted according to the specified format string. Returns the current datetime in the requested format.",
  "input_schema": {
    "type": "object",
    "properties": {
      "date_format": {
        "type": "string",
        "description": "Python strftime format string (e.g., '%Y-%m-%d %H:%M:%S' for '2024-12-16 14:30:45', '%Y-%m-%d' for date only, '%H:%M:%S' for time only). Defaults to '%Y-%m-%d %H:%M:%S' if not specified.",
        "default": "%Y-%m-%d %H:%M:%S"
      }
    },
    "required": []
  }
})
# get_current_datetime("%H:%M")

In [27]:
messages = []

messages.append(
    {
        "role": "user",
        "content": "What is the exact time, formatted as HH:MM:SS?"
    }
)

response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages = messages,
    tools=[get_current_datetime_schema],
)


# messages.append({
#     "role": "assistant",
#     "content": response.content
# })
add_assistant_message(messages, response)
messages

[{'role': 'user', 'content': 'What is the exact time, formatted as HH:MM:SS?'},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_013zyueiZ3NwLx9SDJicVmM9', input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use')]}]

In [23]:
result = get_current_datetime(**response.content[0].input)

messages.append({
    "role": "user",
    "content": [
        {
            "type": "tool_result",
            "tool_use_id": response.content[0].id,
            "content": result,
            "is_error": False
        }
    ],
    
})
messages

[{'role': 'user', 'content': 'What is the exact time, formatted as HH:MM:SS?'},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_01BdFD1D6zwFEqFJnBe8dCSd', input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use')]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01BdFD1D6zwFEqFJnBe8dCSd',
    'content': '21:00:30',
    'is_error': False}]}]

In [24]:
client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_schema],
)

Message(id='msg_01Dc9wBbZVNj4Ei4VDomFybW', content=[TextBlock(citations=None, text='The exact time is **21:00:30** (9:00:30 PM).', type='text')], model='claude-haiku-4-5-20251001', role='assistant', stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, input_tokens=770, output_tokens=23, server_tool_use=None, service_tier='standard'))

In [47]:
messages = []

# add_user_message(
#     messages,
#     "What is the current time in HH:MM format? Also, what is the current time in SS format?"
# )


add_user_message(
    messages,
    "Set a reminder for my doctors appointment. Its 177 days after Jan 1st, 2050"
)

run_conversation(messages)

I'll calculate when that date is and set a reminder for you.
Now I'll set the reminder for your doctor's appointment:
----
 setting the following reminder for 2050-06-27:
Doctor's appointment
------
Perfect! I've set a reminder for your doctor's appointment on **Monday, June 27, 2050** at 12:00 AM. This is 177 days after January 1st, 2050.


[{'role': 'user',
  'content': 'Set a reminder for my doctors appointment. Its 177 days after Jan 1st, 2050'},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text="I'll calculate when that date is and set a reminder for you.", type='text'),
   ToolUseBlock(id='toolu_01GjLHxc2rFCLYcgxGY3DvMT', input={'datetime_str': '2050-01-01', 'duration': 177, 'unit': 'days', 'input_format': '%Y-%m-%d'}, name='add_duration_to_datetime', type='tool_use')]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01GjLHxc2rFCLYcgxGY3DvMT',
    'content': '"Monday, June 27, 2050 12:00:00 AM"',
    'is_error': False}]},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text="Now I'll set the reminder for your doctor's appointment:", type='text'),
   ToolUseBlock(id='toolu_011veDxs3T5ZsWZYMvsrhkNJ', input={'content': "Doctor's appointment", 'timestamp': '2050-06-27'}, name='set_reminder', type='tool_use')]},
 {'role': 'user',
  'content': [{'type': 't

In [54]:
# demo of multiple tool calls in a single claude message
messages = []

add_user_message(
    messages,
    """
    Set two reminder for Jan 1, 2025 at 8 AM:
        * I have a doctors appointment
        * Taxes are due
    """
)
run_conversation(messages)


----
 setting the following reminder for 2025-01-01 08:00:00:
I have a doctors appointment
------
----
 setting the following reminder for 2025-01-01 08:00:00:
Taxes are due
------
Perfect! I've set both reminders for January 1, 2025 at 8:00 AM:
1. ✓ "I have a doctors appointment"
2. ✓ "Taxes are due"

Both reminders are now scheduled and will trigger at the specified time.


[{'role': 'user',
  'content': '\n    Set two reminder for Jan 1, 2025 at 8 AM:\n        * I have a doctors appointment\n        * Taxes are due\n    '},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_012VKeZZq1Kgvj8WYxd8o3m8', input={'content': 'I have a doctors appointment', 'timestamp': '2025-01-01 08:00:00'}, name='set_reminder', type='tool_use'),
   ToolUseBlock(id='toolu_01QMvUjnH22XKrrtbjXh6VzM', input={'content': 'Taxes are due', 'timestamp': '2025-01-01 08:00:00'}, name='set_reminder', type='tool_use')]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_012VKeZZq1Kgvj8WYxd8o3m8',
    'content': 'null',
    'is_error': False},
   {'type': 'tool_result',
    'tool_use_id': 'toolu_01QMvUjnH22XKrrtbjXh6VzM',
    'content': 'null',
    'is_error': False}]},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text='Perfect! I\'ve set both reminders for January 1, 2025 at 8:00 AM:\n1. ✓ "I have a doctors appointment"\n2. ✓ "Tax

In [59]:
messages = []

add_user_message(
    messages,
    """
    Write a one-paragraph scholarly article about computer science.
    Include a title and author name.
    """
)



response = chat(messages)
text_from_message(response)

"# The Evolution of Quantum Computing: From Theoretical Framework to Practical Implementation\n\n**Dr. Sarah Mitchell, Department of Computer Science, Stanford University**\n\nQuantum computing represents a paradigm shift in computational theory, leveraging the principles of quantum mechanics—particularly superposition and entanglement—to process information in fundamentally different ways than classical computers. While theoretical foundations for quantum computation were established in the 1980s through the work of pioneers such as Richard Feynman and David Deutsch, significant progress toward practical quantum computers has accelerated in recent years, with organizations including IBM, Google, and IonQ developing increasingly sophisticated quantum processors. Despite remarkable achievements, such as Google's 2019 demonstration of quantum supremacy, substantial challenges remain, including qubit decoherence, error correction, and the development of quantum algorithms with real-world 

In [70]:
messages = []
add_user_message(messages, text_from_message(response))
response = chat(
    messages,
    tools=[article_summary_schema],
    tool_choice={"type": "tool", "name": "article_summary"},
)
print(response)
response.content[0].input

BadRequestError: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'messages.0: all messages must have non-empty content except for the optional final assistant message'}, 'request_id': 'req_011CWcC8Sc7zo6M4du5rDRem'}